# IrodoriTTS Studio for Colab

IrodoriTTSをColab上で起動し、ブラウザGUIから連続音声生成するためのノートブックです。

## 最初にやること

上部メニューから以下を設定してください。

 →  →  →  → 

T4 GPUでOKです。CPUでも起動できる場合がありますが、音声生成にかなり時間がかかるためGPU推奨です。


## STEP 1：GPU確認

 でGPU名が表示されればOKです。
エラーが出る場合は、ランタイムがGPUになっていません。


In [ ]:
!nvidia-smi


## STEP 2：Google Driveをマウントする（任意）

生成した音声や  を残したい場合は、Google Driveをマウントしてください。
一時的に試すだけなら、このセルはスキップしても大丈夫です。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## STEP 3：Irodori-TTS本体とStudio GUIを取得

公式Irodori-TTS本体と、このStudio GUIリポジトリをColab上にcloneします。

生成ファイルはStudio側の `outputs` に保存します。


In [ ]:
%cd /content
!rm -rf Irodori-TTS irodori-tts-studio
!git clone https://github.com/Aratako/Irodori-TTS.git
!git clone https://github.com/goroyattemiyo/irodori-tts-studio.git
!ls -ld /content/Irodori-TTS /content/irodori-tts-studio


## STEP 4：環境構築

ffmpeg、uv、pydub、FastAPI関連を入れて、Irodori-TTSの依存関係をセットアップします。

初回は時間がかかります。


In [ ]:
%cd /content/Irodori-TTS
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -U uv pydub fastapi uvicorn python-multipart
!uv sync --extra cu128


## STEP 5：Studio Web UIを起動

Studio側のFastAPI Web UIを起動します。

このセルでは cloudflared を使って、Colab外から開ける一時URLを表示します。

`https://xxxxx.trycloudflare.com` のようなURLが表示されたら、それを開いてGUIを使用してください。

停止したい場合は、このセルの実行を停止します。


In [ ]:
import os
import re
import subprocess
import time

os.environ["IRODORI_TTS_ROOT"] = "/content/Irodori-TTS"
os.environ["IRODORI_OUTPUT_ROOT"] = "/content/irodori-tts-studio/outputs"

%cd /content/irodori-tts-studio

# cloudflared を準備して、FastAPI用の公開URLを作ります
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(80):
    line = tunnel.stdout.readline()
    if line:
        print(line, end="")
        match = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n==============================")
    print("Studio Web UI URL:", public_url)
    print("==============================\n")
else:
    print("公開URLを取得できませんでした。ログを確認してください。")

# FastAPI Web UIを起動します
!python -m app.api_server


## 保存について

Colabの `/content` 配下は、セッション終了後に消える場合があります。

残したい場合は、生成後にStudio側の `outputs` フォルダをGoogle Driveへコピーしてください。


In [ ]:
# Google Driveへ保存したい場合に実行
!mkdir -p /content/drive/MyDrive/IrodoriTTS_projects
!cp -r /content/irodori-tts-studio/outputs /content/drive/MyDrive/IrodoriTTS_projects/
